# 🏈 NFL PlayVision
### Real tracking · rolling motion forecasts · interactive receiver–defender replay

Buka `PlayVision_Offline.html` langsung untuk pengalaman offline tanpa instalasi. Notebook ini menampilkan hasil terverifikasi dan dapat melatih ulang model.

**Lingkup:** prediksi posisi 0,5/1 detik, bukan hasil pertandingan atau catch probability. Data model bawaan berasal dari 12 pertandingan 2021. Tiga replay lengkap 2017 merupakan ilustrasi dari era lain. Data 2026 Kaggle belum dilatih di sini.

In [1]:
from pathlib import Path
import sys,os,zipfile
IN_COLAB='google.colab' in sys.modules or Path('/content').exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP_PATH=Path('/content/drive/MyDrive/NFL_PlayVision.zip')
    if not ZIP_PATH.exists():raise FileNotFoundError('Upload NFL_PlayVision.zip ke My Drive, atau ubah ZIP_PATH.')
    destination=Path('/content/playvision_workspace');destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        for name in z.namelist():
            if not (destination/name).resolve().is_relative_to(destination.resolve()):raise ValueError('Unsafe ZIP path')
        z.extractall(destination)
    PROJECT=destination/'nfl_playvision'
else:
    PROJECT=Path.cwd()
    if not (PROJECT/'src/playvision.py').exists():PROJECT=PROJECT/'nfl_playvision'
PROJECT=PROJECT.resolve()
assert (PROJECT/'src/playvision.py').exists()
os.chdir(PROJECT)
print('Project:',PROJECT)

Project: /workspace/scratch/fb3c2bf1d048/nfl_playvision


In [2]:
import subprocess
if IN_COLAB:subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
# Jika diminta restart runtime, restart lalu jalankan kembali sel setup.
TRAIN_NEW=False
RUN_TESTS=False
BUDGET_PER_AXIS=60
MAX_TRAIN_SAMPLES=100000
import json,datetime,shutil
import pandas as pd
import plotly.express as px
from IPython.display import display,HTML,Markdown
sys.path.insert(0,str(PROJECT/'src'))
from playvision import download,prepare,train,evaluate,predict
RUN=PROJECT/'models/verified' 

## 1 · Replay offline
Semua posisi berasal dari tracking nyata. Future truth hanya overlay evaluasi, tidak menjadi input model. Panel nearest DB adalah hubungan geometris, bukan assignment coverage.

Aplikasi berikut adalah demo bawaan; sumber adegan selalu ditampilkan. Jika tampilan iframe dibatasi oleh browser Colab, unduh dan buka HTML langsung.

In [3]:
import html
page=(PROJECT/'PlayVision_Offline.html').read_text()
frame='<iframe title="NFL PlayVision" style="width:100%;height:1100px;border:0" sandbox="allow-scripts" srcdoc="'+html.escape(page,quote=True)+'"></iframe>'
display(HTML(frame))

/root/.local/lib/python3.12/site-packages/IPython/core/display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


[Offline replay HTML executed; payload in PlayVision_Offline.html]

## 2 · Data dan split
Delapan game latihan minggu 1–2, dua validasi minggu 3, dua tes minggu 4. Tidak ada game yang muncul di lebih dari satu split. Frame berdekatan tetap berkorelasi, sehingga jumlah frame bukan jumlah pertandingan independen.

Regional data berakhir 0,3–0,5 detik setelah release; angka regional terutama menggambarkan fase sebelum release dalam passing play. Lihat phase metrics dan evaluasi 2017 terpisah.

In [4]:
plan=json.loads((PROJECT/'data/split_plan.json').read_text())
display(pd.DataFrame([{'gameId':g,'split':s} for g,s in plan.items()]))
if RUN_TESTS:
    subprocess.run([sys.executable,'-m','pytest','tests','-q'],check=True)

,gameId,split
0,2021090900,train
1,2021091200,train
2,2021091201,train
3,2021091202,train
4,2021091600,train
5,2021091900,train
6,2021091901,train
7,2021091902,train
8,2021092300,validation
9,2021092600,validation


## 3 · Pelatihan ulang (opsional)
Fitur menggunakan riwayat dan tetangga pada frame saat ini saja. AutoML belajar koreksi dx/dy terhadap baseline kecepatan tetap. Validasi memilih baseline/campuran untuk setiap horizon. Test dibuka setelah model dibekukan.

Mengulang pelatihan pada split tes yang sama tidak menghasilkan tes independen baru. CPU cukup. Budget bukan batas waktu total keras.

In [5]:
if TRAIN_NEW:
    download();prepare()
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    RUN=PROJECT/'runs'/f'colab_{stamp}'
    train(BUDGET_PER_AXIS,MAX_TRAIN_SAMPLES,run_dir=RUN)
    evaluate(run_dir=RUN)
    if IN_COLAB:
        saved=Path('/content/drive/MyDrive/NFL_PlayVision_runs')/RUN.name
        shutil.copytree(RUN,saved,dirs_exist_ok=True)
        shutil.copytree(PROJECT/'data',Path('/content/drive/MyDrive/NFL_PlayVision_runs/data_snapshot'),dirs_exist_ok=True)
        print('Saved:',saved)
print('Using metrics:',RUN)

Using metrics: /workspace/scratch/fb3c2bf1d048/nfl_playvision/models/verified


## 4 · Ukuran hasil yang bermakna
Error diukur dalam yards. Persentase dalam toleransi 1 yard bukan akurasi kemenangan, catch, atau produksi mingguan. Bandingkan pula baseline percepatan tetap yang lebih kuat.

In [6]:
metrics=json.loads((RUN/'metrics.json').read_text());rows=[]
for h,record in metrics['horizons'].items():
    for name in ['constant_velocity','constant_acceleration','selected_model']:
        m=record[name]
        rows.append({'horizon_seconds':h,'model':name,'n':m['n'],'mean_error_yards':m['mean_distance_yards'],
           'within_half_yard_pct':m['within_0_5_yards_pct'],'within_one_yard_pct':m['within_1_yard_pct']})
summary=pd.DataFrame(rows);display(summary.round(3))
px.bar(summary,x='horizon_seconds',y='mean_error_yards',color='model',barmode='group',template='plotly_white',title='Held-out game error: lower is better').show()
print('Test games:',metrics['test_games'],'plays:',metrics['test_plays'])

Test games: [2021093000, 2021100300] plays: 116


,horizon_seconds,model,n,mean_error_yards,within_half_yard_pct,within_one_yard_pct
0,0.5,constant_velocity,16799,0.607,41.538,86.702
1,0.5,constant_acceleration,16799,0.372,73.332,98.940
2,0.5,selected_model,16799,0.251,93.410,99.970
3,1.0,constant_velocity,13886,1.835,7.259,21.388
4,1.0,constant_acceleration,13886,1.369,13.589,38.089
5,1.0,selected_model,13886,0.919,24.226,61.602


## 5 · Inferensi tanpa koordinat masa depan
Contoh berisi fitur yang sudah dihitung dari frame saat ini dan sejarahnya. `build_inputs` dapat membuat fitur dari prefix observasi saja tanpa membutuhkan future labels.

In [7]:
import joblib
bundle=joblib.load(RUN/'model.joblib')
example=pd.read_csv(PROJECT/'examples/forecast_inputs.csv')
dx,dy=predict(bundle,example)
example['predicted_delta_x']=dx;example['predicted_delta_y']=dy
display(example[['gameId','playId','actor_id','frameId','horizon','predicted_delta_x','predicted_delta_y']].head())

/opt/codex/runtimes/codex-primary-runtime/dependencies/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/codex/runtimes/codex-primary-runtime/dependencies/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,gameId,playId,actor_id,frameId,horizon,predicted_delta_x,predicted_delta_y
0,2021093000,209,38696,10,0.5,1.243763,0.293211
1,2021093000,209,38696,12,0.5,1.935925,0.385439
2,2021093000,209,38696,14,0.5,2.442620,0.450283
3,2021093000,209,38696,16,0.5,2.775536,0.630824
4,2021093000,209,38696,18,0.5,2.701743,0.685448


## 6 · Evaluasi eksternal dan batasan
Angka berikut adalah hasil model bawaan pada satu game 2017, termasuk fase setelah release. Bukan angka run baru jika Anda retrain. Perbedaan era dan satu game membatasi klaim generalisasi.

Untuk data Kaggle 2026, baca KAGGLE_2026_GUIDE.md. Importer membuang future landing coordinates dan role target. Adapter schema diuji dengan fixture, belum dengan dataset kompetisi asli.

In [8]:
external=json.loads((PROJECT/'reports/external2017_metrics.json').read_text())
display(pd.DataFrame([{'horizon':h,**m['post_release']} for h,m in external['horizons'].items()]))
display(Markdown((PROJECT/'reports/RESULTS.md').read_text()))
if IN_COLAB:
    save_root=Path('/content/drive/MyDrive/NFL_PlayVision_runs');save_root.mkdir(exist_ok=True)
    summary.to_csv(save_root/'metrics_summary.csv',index=False)
    shutil.copy2(PROJECT/'PlayVision_Offline.html',save_root/'PlayVision_Offline.html')
    print('Saved:',save_root)

,horizon,n,mean_distance_yards,median_distance_yards,p90_distance_yards,coordinate_RMSE_yards,within_0_5_yards_pct,within_1_yard_pct,within_2_yards_pct
0,0.5,4612,0.314492,0.280541,0.561111,0.258177,84.995663,99.588031,100.000000
1,1.0,4612,0.972311,0.895508,1.693625,0.784549,19.796184,58.152645,95.490026


# Hasil terverifikasi NFL PlayVision

Prediksi perpindahan posisi pemain 0,5 dan 1 detik dari riwayat tracking, bukan prediksi hasil pertandingan.

| Horizon | Error rata-rata (yard) | Dalam 0,5 yard | Dalam 1 yard | Baseline percepatan: error |
|---|---:|---:|---:|---:|
| 0,5 detik | 0,251 | 93,41% | 99,97% | 0,372 |
| 1 detik | 0,919 | 24,23% | 61,60% | 1,369 |

Tes regional 2021 mencakup 2 pertandingan dan 116 passing plays, terpisah dari 8 game latihan dan 2 validasi. Jumlah sampel per horizon 16.799 dan 13.886; sampel dalam satu play berkorelasi. Error model lebih rendah sekitar 32–33% dari baseline percepatan tetap.

Data regional berakhir hanya 0,3–0,5 detik setelah release. Mayoritas evaluasi adalah gerakan sebelum release pada passing play; tidak ada sampel regional 1 detik dengan anchor setelah release. Angka tinggi pada horizon pendek tidak boleh dipasarkan sebagai akurasi menang, catch, atau performa mingguan.

Evaluasi tambahan pada satu game lengkap KC–NE tahun 2017, tanpa melatih ulang model: fase setelah release memiliki error rata-rata 0,314 yard (0,5 detik) dan 0,972 yard (1 detik); dalam 1 yard masing-masing 99,59% dan 58,15%. Ini uji transfer era pada satu game, bukan bukti generalisasi liga. Replay touchdown dipilih sebagai ilustrasi cerita, bukan sampel acak untuk evaluasi.

AutoML FLAML mencari LightGBM dan ExtraTrees, budget 60 detik per sumbu. Model akhir LightGBM memprediksi koreksi terhadap kecepatan tetap; campuran baseline ditentukan menggunakan validasi saja. Test dibuka setelah pembekuan model. Fitur input terbatas pada posisi/gerakan yang telah diamati dan geometri tetangga saat ini; future truth hanya untuk target/evaluasi.

Belum dikerjakan: pelatihan pada dataset kompetisi 2026 asli, evaluasi banyak musim/game penuh, interval prediksi terkalibrasi, identifikasi coverage assignment, dan video computer vision. Adapter 2026 diuji dengan fixture schema saja. Angka ini bukan skor leaderboard.
